This script spatially joins scalar field data (roughness, curvature) from a 
full anvil point cloud onto damaged and undamaged coordinate sets.

Input files (per anvil folder), identified by filename keyword:
- Full anvil point cloud: filename contains "spatial_join"
- Damaged coordinates:    filename contains "damage" 
- Undamaged coordinates:  filename contains "undamaged"

For each anvil folder, it:
1. Loads the full-anvil point cloud (with roughness/curvature values), the 
   damaged coordinates, and the undamaged coordinates.
2. Matches each damaged coordinate to its single nearest point on the full 
   anvil surface, carrying over that point's roughness/curvature values.
3. Matches each undamaged coordinate to all full-anvil points within a fixed 
   radius (5.64 mm), carrying over roughness/curvature for each nearby point 
   (so one undamaged coordinate can expand into several output rows).
4. Combines the damaged and undamaged results, labels them by condition, and 
   saves one output file per anvil, named "{anvil_id}_output.txt".

In [ ]:
from pathlib import Path
import pandas as pd
from scipy.spatial import cKDTree

# Paths
root_path  = Path("PATH")   # folder containing one subfolder per anvil

roughness_col = "Roughness (10)"
curvature_col = "Mean curvature (10)"
radius_mm     = 5.64

# Helper functions
def load_txt_with_header(path):
    with open(path, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("//"):
            col_names = [c.strip() for c in line.lstrip("/").strip().split(";")]
            df = pd.read_csv(path, sep=";", skiprows=i + 1, header=None, names=col_names)
            df.columns = [c.lstrip("/") for c in df.columns]
            return df
    raise ValueError(f"No '//' header found in {path}")

def load_txt_no_header(path):
    df = pd.read_csv(path, sep=",", header=None, names=["X", "Y", "Z"])
    return df

def save_txt(df, path):
    with open(path, "w") as f:
        f.write("//" + ";".join(df.columns) + "\n")
        df.to_csv(f, index=False, header=False, sep=";")

# Process each anvil subfolder
anvil_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])
print(f"Found {len(anvil_dirs)} anvil folder(s)")

for anvil_dir in anvil_dirs:
    anvil_id = anvil_dir.name
    print(f"\n{'='*60}")
    print(f"Processing: {anvil_id}")
    print(f"{'='*60}")

    txt_files = list(anvil_dir.glob("*.txt"))

    # Identify files by keyword
    full_anvil_files = [f for f in txt_files if "spatial_join" in f.name.lower()]
    damaged_files    = [f for f in txt_files if "damage" in f.name.lower() and "undamage" not in f.name.lower()]
    undamaged_files  = [f for f in txt_files if "undamage" in f.name.lower()]

    # Validate
    if not full_anvil_files:
        print(f"  Skipping — no spatial_join file found")
        continue
    if not damaged_files:
        print(f"  Skipping — no damaged file found")
        continue
    if not undamaged_files:
        print(f"  Skipping — no undamaged file found")
        continue

    full_anvil_txt = full_anvil_files[0]
    damaged_txt    = damaged_files[0]
    undamaged_txt  = undamaged_files[0]

    print(f"  Full anvil:  {full_anvil_txt.name}")
    print(f"  Damaged:     {damaged_txt.name}")
    print(f"  Undamaged:   {undamaged_txt.name}")

    # Load files
    print("  Loading files...")
    df_full      = load_txt_with_header(full_anvil_txt)
    df_damaged   = load_txt_with_header(damaged_txt)
    df_undamaged = load_txt_no_header(undamaged_txt)

    print(f"    Full anvil:  {len(df_full):,} points")
    print(f"    Damaged:     {len(df_damaged):,} points")
    print(f"    Undamaged:   {len(df_undamaged):,} points")

    # Verify scalar columns exist in full anvil
    for col in [roughness_col, curvature_col]:
        if col not in df_full.columns:
            print(f"  Skipping — '{col}' not found in full anvil. Available: {list(df_full.columns)}")
            continue

    # Build KD-tree on full anvil XYZ
    print("  Building KD-tree on full anvil...")
    full_xyz = df_full[["X", "Y", "Z"]].to_numpy(dtype=float)
    tree = cKDTree(full_xyz)

    # Damaged: nearest neighbour match (1 row per coordinate)
    print("  Matching damaged coordinates (nearest neighbour)...")
    damaged_xyz = df_damaged[["X", "Y", "Z"]].to_numpy(dtype=float)
    _, idx = tree.query(damaged_xyz, k=1)

    df_damaged_out = df_damaged[["X", "Y", "Z"]].copy()
    df_damaged_out[roughness_col] = df_full[roughness_col].values[idx]
    df_damaged_out[curvature_col] = df_full[curvature_col].values[idx]
    df_damaged_out["condition"]   = "damaged"
    print(f"    Output rows: {len(df_damaged_out):,}")

    # Undamaged: all full anvil points within radius_mm
    print(f"  Matching undamaged coordinates (radius query, r={radius_mm}mm)...")
    undamaged_xyz = df_undamaged[["X", "Y", "Z"]].to_numpy(dtype=float)
    neighbours = tree.query_ball_point(undamaged_xyz, r=radius_mm)

    records = []
    for i, neighbour_idx in enumerate(neighbours):
        if len(neighbour_idx) == 0:
            print(f"    Warning: no full anvil points within {radius_mm}mm of undamaged point {i} — skipping")
            continue
        for j in neighbour_idx:
            records.append({
                "X":           full_xyz[j, 0],
                "Y":           full_xyz[j, 1],
                "Z":           full_xyz[j, 2],
                roughness_col: df_full[roughness_col].values[j],
                curvature_col: df_full[curvature_col].values[j],
                "condition":   "undamaged",
            })

    df_undamaged_out = pd.DataFrame.from_records(records)
    print(f"    Input rows:  {len(df_undamaged):,}")
    print(f"    Output rows: {len(df_undamaged_out):,} ({len(df_undamaged_out) / len(df_undamaged):.1f}x expansion)")

    # Combine and save
    print("  Combining and saving...")
    df_out = pd.concat([df_damaged_out, df_undamaged_out], ignore_index=True)
    print(f"    Total output rows: {len(df_out):,}")

    output_path = anvil_dir / f"{anvil_id}_output.txt"
    save_txt(df_out, output_path)
    print(f"    Saved to: {output_path}")

print(f"\nAll done.")